## Bronze to Silver

In [21]:
# JSON TO CSV

import json
import csv
from pathlib import Path


def json_to_csv(
    file_prefix,
    output_csv,
    record_extractor
):

    input_dir = "../data/bronze"
    output_csv = Path(output_csv)

    page = 1
    fieldnames = None

    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = None

        while True:
            json_file = Path(input_dir + f"/{file_prefix}_page_{page}.json")
            if not json_file.exists():
                ###CHECK###
                print("DOES NOT EXIST: ", json_file)
                break

            with open(json_file, "r", encoding="utf-8") as jf:
                data = json.load(jf)

            rows = record_extractor(data)

            if not rows:
                page += 1
                continue

            # Initialize CSV writer on first non-empty page
            if writer is None:
                fieldnames = rows[0].keys()
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()

            writer.writerows(rows)
            page += 1


In [29]:
# Extracters specific to data

def extract_members(json_data):
    rows = []
    for m in json_data.get("members", []):
        # Get first term if it exists
        term = m.get("terms", {}).get("item", [{}])[0]
        rows.append({
            "bioguide_id": m.get("bioguideId"),
            "name": m.get("name"),
            "first_name": m.get("name").split(",")[1].strip() if m.get("name") else None,
            "last_name": m.get("name").split(",")[0].strip() if m.get("name") else None,
            "party": m.get("partyName"),
            "state": m.get("state"),
            "district": m.get("district"),
            "chamber": term.get("chamber"),
            "term_start_year": term.get("startYear"),
            "depiction_url": m.get("depiction", {}).get("imageUrl"),
            "update_date": m.get("updateDate"),
            "member_url": m.get("url")
        })
    return rows

def extract_house_rollcalls(json_data):
    rows = []
    for v in json_data.get("houseRollCallVotes", []):
        rows.append({
            "congress": v.get("congress"),
            "session": v.get("sessionNumber"),
            "roll_call_number": v.get("rollCallNumber"),
            "identifier": v.get("identifier"),
            "legislation_number": v.get("legislationNumber"),
            "legislation_type": v.get("legislationType"),
            "legislation_url": v.get("legislationUrl"),
            "result": v.get("result"),
            "vote_type": v.get("voteType"),
            "source_data_url": v.get("sourceDataURL"),
            "start_date": v.get("startDate"),
            "update_date": v.get("updateDate"),
            "api_url": v.get("url")
        })
    return rows

def extract_bills(json_data):
    rows = []
    for b in json_data.get("bills", []):
        latest_action = b.get("latestAction") or {}  # <-- fix here
        rows.append({
            "congress": b.get("congress"),
            "bill_number": b.get("number"),
            "bill_type": b.get("type"),
            "title": b.get("title"),
            "origin_chamber": b.get("originChamber"),
            "origin_chamber_code": b.get("originChamberCode"),
            "latest_action_date": latest_action.get("actionDate"),
            "latest_action_text": latest_action.get("text"),
            "update_date": b.get("updateDate"),
            "bill_url": b.get("url")
        })
    return rows

def extract_amendments(json_data):
    rows = []
    for a in json_data.get("amendments", []):
        latest_action = a.get("latestAction", {})
        rows.append({
            "congress": a.get("congress"),
            "amendment_number": a.get("number"),
            "amendment_type": a.get("type"),
            "description": a.get("description"),
            "latest_action_date": latest_action.get("actionDate"),
            "latest_action_time": latest_action.get("actionTime"),
            "latest_action_text": latest_action.get("text"),
            "update_date": a.get("updateDate"),
            "amendment_url": a.get("url")
        })
    return rows


In [27]:
# Amendments data
json_to_csv(
    file_prefix="amendments",
    output_csv="../data/silver/amendments_119.csv",
    record_extractor=extract_amendments
)

DOES NOT EXIST:  ..\data\bronze\amendments_page_19.json


In [30]:
# Bills data

json_to_csv(
    file_prefix="bills",
    output_csv="../data/silver/bills_119.csv",
    record_extractor=extract_bills
)


DOES NOT EXIST:  ..\data\bronze\bills_page_54.json


In [26]:
# House rollcall data

json_to_csv(
    file_prefix="house_rollcall",
    output_csv="../data/silver/house_rollcall_119.csv",
    record_extractor=extract_house_rollcalls
)


DOES NOT EXIST:  ..\data\bronze\house_rollcall_page_3.json


In [25]:
# Members data

json_to_csv(
    file_prefix="members",
    output_csv="../data/silver/members_119.csv",
    record_extractor=extract_members
)

DOES NOT EXIST:  ..\data\bronze\members_page_4.json


In [ ]:
input_dir = "../data/bronze"
file_prefix="house_roll_call"
page=1

json_file = input_dir + f"/{file_prefix}_page_{page}.json"
print(json_file)


In [2]:
# XML TO CSV

import csv
import xml.etree.ElementTree as ET
from pathlib import Path


def senate_rollcall_xml_to_csv(xml_files, output_csv):
    rows = []

    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # top-level metadata (same for all votes in file)
        congress = root.findtext("congress")
        session = root.findtext("session")
        congress_year = root.findtext("congress_year")

        for vote in root.findall("./votes/vote"):
            row = {
                "congress": congress,
                "session": session,
                "congress_year": congress_year,
                "vote_number": vote.findtext("vote_number"),
                "vote_date": vote.findtext("vote_date"),
                "issue": vote.findtext("issue"),
                "question": vote.findtext("question").strip()
                            if vote.findtext("question") else None,
                "result": vote.findtext("result"),
                "yeas": vote.findtext("vote_tally/yeas"),
                "nays": vote.findtext("vote_tally/nays"),
                "title": vote.findtext("title")
            }
            rows.append(row)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "congress",
                "session",
                "congress_year",
                "vote_number",
                "vote_date",
                "issue",
                "question",
                "result",
                "yeas",
                "nays",
                "title"
            ]
        )
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
# For Senate roll call data
xml_files = ["../data/bronze/senate_rollcall_page_1.xml", "../data/bronze/senate_rollcall_page_2.xml"]

senate_rollcall_xml_to_csv(
    xml_files,
    "../data/silver/senate_rollcall_119.csv"
)
